# 跨字段逻辑质量审计：零播放却有互动

定位播放时长为 0 但发生点赞、评论、关注或转发的记录，评估其规模、集中位置及对核心指标的影响。


## 1. 读取已经验证通过的标准推荐清洗表

只读取 `data/processed/log_standard_clean.parquet`。这里使用现有清洗结果，不重新清洗，也不修改 Parquet 文件。


In [17]:
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """从当前目录向上定位作品集根目录。"""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Python").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请从 KuaiRand_Pure 目录或其子目录运行。")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MYSQL_IMPORT_DIR = PROJECT_ROOT / "data" / "mysql_import"

data_path = PROCESSED_DIR / "log_standard_clean.parquet"
if not data_path.exists():
    raise FileNotFoundError(f"缺少清洗文件：{data_path}")

df = pd.read_parquet(data_path)
print("读取文件：", data_path)
print("数据形状：", df.shape)
df.head()


读取文件： E:\Users\yanyan\Desktop\KuaiRand_Pure\data\processed\log_standard_clean.parquet
数据形状： (1414622, 25)


,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,...,comment_stay_time,is_profile_enter,is_rand,tab,is_duration_missing,duration_ms_clean,play_ratio_raw,is_complete_play,date_clean,log_source
0,0,1527,20220411,1900,1649675512388,0,0,0,0,0,...,0,0,0,1,0,209900.0,0.006598,0,2022-04-11,standard_0408_0421
1,0,7405,20220416,2000,1650111976017,0,0,0,0,0,...,0,0,0,0,0,65400.0,0.000000,0,2022-04-16,standard_0408_0421
2,0,6026,20220420,1600,1650444367095,0,0,0,0,0,...,0,0,0,1,0,170833.0,0.008224,0,2022-04-20,standard_0408_0421
3,1,6354,20220411,1100,1649645295928,0,0,0,0,0,...,0,0,0,8,0,255160.0,0.000000,0,2022-04-11,standard_0408_0421
4,1,3645,20220411,1100,1649648827559,0,0,0,0,0,...,0,0,0,1,0,79733.0,0.024707,0,2022-04-11,standard_0408_0421


## 2. 定义互动字段并建立审计标记

`interaction_cols` 是需要检查的二元互动字段。`has_interaction` 表示一行中至少有一种互动；`is_zero_play` 表示播放时长等于0。根据前面的基础检查，本项目的播放时长没有负数，因此这里只审计零播放记录。


In [18]:
interaction_cols = ['is_like', 'is_comment', 'is_follow', 'is_forward']
required_cols = interaction_cols + ['play_time_ms', 'user_id', 'video_id', 'date_clean']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f'缺少字段：{missing_cols}')

df['has_interaction'] = df[interaction_cols].eq(1).any(axis=1)  #按行来看
df['is_zero_play'] = df['play_time_ms'].eq(0)
df['interaction_without_play'] = (
    df['has_interaction'] & df['is_zero_play']
).astype('int8')

print('互动字段：', interaction_cols)
print('新增审计标记：has_interaction、is_zero_play、interaction_without_play')


互动字段： ['is_like', 'is_comment', 'is_follow', 'is_forward']
新增审计标记：has_interaction、is_zero_play、interaction_without_play


## 3. 总体规模：零播放互动记录？

同时输出记录数和占比。这里的主要分母是全部标准推荐记录；另外单独计算它占全部互动记录的比例。


In [19]:
total_records = len(df)
interaction_records = int(df['has_interaction'].sum())
zero_play_interaction_records = int((df['has_interaction'] & df['is_zero_play']).sum())

overall_summary = pd.DataFrame({
    '检查项': [
        '标准推荐记录',
        '至少一种互动的记录',
        '互动且播放时长=0'
    ],
    '记录数': [
        total_records,
        interaction_records,
        zero_play_interaction_records
    ]
})
overall_summary['占标准推荐记录比例(%)'] = (
    overall_summary['记录数'] / total_records * 100
).round(4)
overall_summary


,检查项,记录数,占标准推荐记录比例(%)
0,标准推荐记录,1414622,100.0000
1,至少一种互动的记录,31445,2.2229
2,互动且播放时长=0,376,0.0266


In [20]:
print('互动且播放时长=0，占全部互动记录比例：', round(zero_play_interaction_records / interaction_records * 100, 4) if interaction_records else 0, '%')
#此为python的条件表达式： 满足条件时的结果 if 条件 else 不满足条件时的结果

互动且播放时长=0，占全部互动记录比例： 1.1957 %


## 4. 按互动类型拆分，是否集中在某种互动类型上？

同一条记录可能同时点赞和评论，因此各互动类型的记录数不能简单相加。这里分别计算每种互动在零播放记录中的规模。


In [21]:
behavior_rows = []
for col in interaction_cols:
    behavior_count = int(df[col].eq(1).sum())
    suspicious_count = int((df[col].eq(1) & df['is_zero_play']).sum())
    behavior_rows.append({
        '互动字段': col,
        '互动记录数': behavior_count,
        '其中播放时长=0记录数': suspicious_count,
        '占该互动记录比例(%)': round(suspicious_count / behavior_count * 100, 4) if behavior_count else 0
    })

behavior_summary = pd.DataFrame(behavior_rows)
behavior_summary


,互动字段,互动记录数,其中播放时长=0记录数,占该互动记录比例(%)
0,is_like,26374,312,1.1830
1,is_comment,3637,30,0.8249
2,is_follow,1516,33,2.1768
3,is_forward,1374,22,1.6012


## 5. 按日期定位：是否集中在某些日期？

如果异常只集中在少数日期，可能与某段时间的采集或统计口径有关；如果均匀分布，更像是行为日志之间的常见口径差异。


In [22]:
daily_audit = (
    df.groupby('date_clean', dropna=False)
      .agg(
          推荐记录数=('user_id', 'size'),
          互动记录数=('has_interaction', 'sum'),
          零播放互动记录数=('interaction_without_play', 'sum')
      )
)
daily_audit['零播放互动占比(%)'] = (
    daily_audit['零播放互动记录数'] / daily_audit['推荐记录数'] * 100
).round(4)
daily_audit.sort_values('零播放互动记录数', ascending=False).head(20)


,推荐记录数,互动记录数,零播放互动记录数,零播放互动占比(%)
date_clean,,,,
2022-04-11,276058,5865,97,0.0351
2022-04-10,226258,5061,82,0.0362
2022-04-12,162517,3523,34,0.0209
2022-04-13,92052,2038,21,0.0228
2022-04-15,58326,1570,21,0.0360
2022-04-09,52481,1088,18,0.0343
2022-04-14,69194,1497,16,0.0231
2022-04-16,60592,1712,14,0.0231
2022-04-17,43615,1105,9,0.0206


## 6. 按来源、用户和视频定位

`log_source` 可以判断异常是否集中在某个源文件。用户和视频排名用于观察是否由少数实体贡献了大量异常记录。排名只能描述集中度，不能证明这些用户或视频本身有问题。


In [23]:
if 'log_source' in df.columns:
    source_audit = (
        df.groupby('log_source', dropna=False)
          .agg(
              推荐记录数=('user_id', 'size'),
              互动记录数=('has_interaction', 'sum'),
              零播放互动记录数=('interaction_without_play', 'sum')
          )
    )
    source_audit['零播放互动占比(%)'] = (
        source_audit['零播放互动记录数'] / source_audit['推荐记录数'] * 100
    ).round(4)
    display(source_audit)
else:
    print('当前表没有 log_source 字段，跳过来源分布。')

suspect_df = df.loc[df['interaction_without_play'].eq(1)]
#布尔索引，通过布尔条件筛选符合要求的行
top_users = suspect_df.groupby('user_id').size().sort_values(ascending=False).head(20).rename('异常用户数')
#size：是pandas中的内置函数，旨在统计组内全部行数，包含有空值 NaN 的行。只关心多少行。
#count：统计该列非空不为 NaN 的行数，会跳过 NaN。
top_videos = suspect_df.groupby('video_id').size().sort_values(ascending=False).head(20).rename('异常记录数')
print('异常记录最多的用户：')
display(top_users.to_frame())  # .to_frame()将Series转换成Frame
print('异常记录最多的视频：')
display(top_videos.to_frame())
# .squeeze()可以把只有一列的 DataFrame 变回 Series。


,推荐记录数,互动记录数,零播放互动记录数,零播放互动占比(%)
log_source,,,,
standard_0408_0421,1125503,25249,330,0.0293
standard_0422_0508,289119,6196,46,0.0159


异常记录最多的用户：


,异常用户数
user_id,
17006,4
18083,3
4165,3
9885,3
17968,3
13089,2
8116,2
6908,2
7378,2


异常记录最多的视频：


,异常记录数
video_id,
6687,5
647,4
2303,4
2399,4
6769,3
6069,3
526,3
3587,3
1280,3


## 7. 抽查异常样本

抽查原始字段和清洗字段，确认这些记录是互动字段为1、播放时长为0，还是存在字段转换或标记错误。


In [24]:
sample_cols = [
    col for col in [
        'user_id', 'video_id', 'date_clean', 'hourmin', 'play_time_ms',
        'duration_ms_clean', 'play_ratio_raw', 'is_click', 'long_view',
        'is_complete_play', 'is_like', 'is_comment', 'is_follow',
        'is_forward', 'log_source'
    ] if col in df.columns
]

suspect_df[sample_cols].head(20)


,user_id,video_id,date_clean,hourmin,play_time_ms,duration_ms_clean,play_ratio_raw,is_click,long_view,is_complete_play,is_like,is_comment,is_follow,is_forward,log_source
7768,342,3955,2022-04-11,2100,0,116440.0,0.0,0,0,0,1,0,0,0,standard_0408_0421
8227,361,5872,2022-04-11,1700,0,409233.0,0.0,0,0,0,1,0,0,0,standard_0408_0421
16266,700,3273,2022-04-12,2300,0,108433.0,0.0,1,0,0,1,0,0,0,standard_0408_0421
18471,793,6231,2022-04-11,2200,0,85566.0,0.0,1,0,0,1,0,0,0,standard_0408_0421
21384,901,2210,2022-04-11,1900,0,217222.0,0.0,0,0,0,1,0,0,0,standard_0408_0421
24389,1028,5699,2022-04-11,1700,0,60400.0,0.0,0,0,0,1,0,0,0,standard_0408_0421
24775,1048,3685,2022-04-10,1400,0,47750.0,0.0,1,0,0,1,0,1,0,standard_0408_0421
34459,1493,7136,2022-04-09,2000,0,259066.0,0.0,1,0,0,0,1,0,0,standard_0408_0421
43812,1864,2761,2022-04-17,2100,0,95533.0,0.0,1,0,0,1,0,0,0,standard_0408_0421
46934,1990,1991,2022-04-12,1900,0,138266.0,0.0,1,0,0,1,0,0,0,standard_0408_0421


## 8. 敏感性分析：是否足以影响核心指标？

这里比较两种口径：

- 原始口径：保留全部标准推荐记录；
- 对照口径：仅排除‘播放时长<=0且发生互动’的记录。

对照口径不是新的清洗表，只用于判断结论稳健性。


In [26]:
def metric_snapshot(table, label):
    valid_complete = table['is_complete_play'].notna()
    return {
        '口径': label,
        '推荐记录数': len(table),
        '点击率(%)': table['is_click'].mean() * 100,
        '长播率(%)': table['long_view'].mean() * 100,
        '完整播放率(%)': table.loc[valid_complete, 'is_complete_play'].mean() * 100,
        '点赞率(%)': table['is_like'].mean() * 100,
        '评论率(%)': table['is_comment'].mean() * 100,
        '关注率(%)': table['is_follow'].mean() * 100,
        '转发率(%)': table['is_forward'].mean() * 100,
        '综合互动率(%)': table[interaction_cols].eq(1).any(axis=1).mean() * 100,
        '播放时长中位数(秒)': table['play_time_ms'].median() / 1000,
        '播放进度中位数': table['play_ratio_raw'].median()
    }

all_metrics = metric_snapshot(df, '原始口径')

comparison_df = df.loc[df['interaction_without_play'].eq(0)].copy()
filtered_metrics = metric_snapshot(comparison_df, '排除零播放互动对照口径')
print(type(filtered_metrics))

sensitivity = pd.DataFrame([all_metrics, filtered_metrics]).set_index('口径').T
sensitivity['绝对变化'] = sensitivity['排除零播放互动对照口径'] - sensitivity['原始口径']
sensitivity['绝对变化'] = sensitivity['绝对变化'].round(6)
sensitivity


<class 'dict'>


口径,原始口径,排除零播放互动对照口径,绝对变化
推荐记录数,1.414622e+06,1.414246e+06,-376.000000
点击率(%),4.639812e+01,4.639200e+01,-0.006119
长播率(%),3.355723e+01,3.356615e+01,0.008922
完整播放率(%),1.543295e+01,1.543704e+01,0.004088
点赞率(%),1.864385e+00,1.842819e+00,-0.021566
评论率(%),2.571005e-01,2.550476e-01,-0.002053
关注率(%),1.071664e-01,1.048615e-01,-0.002305
转发率(%),9.712842e-02,9.559864e-02,-0.001530
综合互动率(%),2.222855e+00,2.196860e+00,-0.025996
播放时长中位数(秒),5.007000e+00,5.011000e+00,0.004000


## 9. 审计结论（基于本次运行结果）

### 9.1 异常规模

- 标准推荐记录共 **1,414,622** 条。
- 至少发生一种互动的记录有 **31,445** 条，占全部推荐记录 **2.2229%**。
- 互动且播放时长等于0的记录有 **376** 条，占全部推荐记录 **0.0266%**，占全部互动记录 **1.1957%**。
- 前面的基础检查已经确认播放时长没有负数，因此本次审计只关注‘播放时长=0且发生互动’这一种实际存在的异常组合。

### 9.2 异常的集中位置

按互动类型看，零播放记录占各类互动记录的比例分别为：点赞 **1.1830%**、评论 **0.8249%**、关注 **2.1768%**、转发 **1.6012%**。同一条记录可能同时包含多种互动，因此这些数量不能相加。

按日志来源看，`standard_0408_0421` 有 **330** 条异常，占该来源推荐记录 **0.0293%**；`standard_0422_0508` 有 **46** 条，占 **0.0159%**。前者比例略高，但两个来源都存在，不能简单认定为某一个文件损坏。

按日期看，异常记录最多的日期是 2022-04-11（97条）和 2022-04-10（82条），但占当日推荐记录的比例仍只有 **0.0351%** 和 **0.0362%**。

按用户和视频看，单个用户最多4条、单个视频最多5条，没有明显由少数用户或视频主导的现象。

### 9.3 对核心指标的影响

将这376条记录排除后，与原始口径相比：

- 点击率下降 **0.0061个百分点**；
- 点赞率下降 **0.0216个百分点**；
- 评论率下降 **0.0021个百分点**；
- 关注率下降 **0.0023个百分点**；
- 转发率下降 **0.0015个百分点**；
- 综合互动率下降 **0.0260个百分点**；
- 播放时长中位数仅变化 **0.004秒**，播放进度中位数仅变化 **0.000082**。

因此，这类记录对本项目**总体核心指标**的影响很小，不足以改变主要业务判断。

### 9.4 最终处理决定

本项目**不删除**这376条记录，也不覆盖已经导出的清洗 Parquet。原因是：它们属于跨字段逻辑可疑记录，但没有官方口径证明它们是无效数据；同时，敏感性分析表明删除它们不会实质改变核心结论。

后续分析继续使用完整的标准推荐清洗表，并在项目数据质量说明中记录：标准推荐日志存在少量‘零播放但发生互动’的记录，占全部推荐记录0.0266%，对总体指标影响有限。若分析具体互动行为或极短播放行为，可引用本 Notebook 的审计结果作为限制说明。
